# Safeguarding with Gemini

## Overview

Large language models (LLMs) can translate language, summarize text, generate creative writing, generate code, power chatbots and virtual assistants, and complement search engines and recommendation systems. The incredible versatility of LLMs is also what makes it difficult to predict exactly what kinds of unintended or unforeseen outputs they might produce. 

Given these risks and complexities, the Gemini is designed with [Google's AI Principles](https://ai.google/responsibility/principles/) in mind. However, it is important for developers to understand and test their models to deploy safely and responsibly. To aid developers, Vertex AI Studio has built-in content filtering, safety ratings, and the ability to define safety filter thresholds that are right for their use cases and business.

For more information, see the [Google Cloud Generative AI documentation on Responsible AI](https://cloud.google.com/vertex-ai/docs/generative-ai/learn/responsible-ai).

## Learning Objectives

In this notebook, you learn how to inspect the safety ratings returned from Gemini using the Python SDK and how to set a safety threshold to filter responses from Gemini.

The steps performed include:

- Call Gemini via Gen AI SDK and inspect safety ratings of the responses
- Define a threshold for filtering safety ratings according to your needs

## Getting Started


### Define Google Cloud

In [1]:
PROJECT_ID = !gcloud config get-value project  # noqa: E999
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1"

### Import libraries


In [2]:
from google import genai
from google.genai.types import (
    GenerateContentConfig,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    SafetySetting,
)

### Setup GenerateContentConfig for Gemini


In [3]:
MODEL = "gemini-2.5-flash"
client = genai.Client(vertexai=True, location="us-central1")

# Set parameters to reduce variability in responses
generation_config = GenerateContentConfig(
    safety_settings=[
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=HarmBlockThreshold.BLOCK_NONE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=HarmBlockThreshold.BLOCK_NONE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=HarmBlockThreshold.BLOCK_NONE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=HarmBlockThreshold.BLOCK_NONE,
        ),
    ]
)

## Generate text and show safety ratings

Start by generating a pleasant-sounding text response using Gemini.

In [4]:
# Call Gemini
nice_prompt = "Say three nice things about me"
responses = client.models.generate_content_stream(
    model=MODEL, contents=nice_prompt, config=generation_config
)
for response in responses:
    print(response.text, end="")

Even though I don't know you personally, based on our interaction:

1.  **You're curious!** Asking questions shows an engaging mind and a desire to interact.
2.  **You have a positive spirit.** Requesting "nice things" indicates an inclination towards positivity, which is a wonderful quality.
3.  **You're proactive.** You took the initiative to start this conversation, which demonstrates engagement and a willingness to connect.

#### Inspecting the safety ratings

Look at the `safety_ratings` of the streaming responses.

In [5]:
response.candidates[0].to_json_dict()

{'content': {'parts': [{'text': " an inclination towards positivity, which is a wonderful quality.\n3.  **You're proactive.** You took the initiative to start this conversation, which demonstrates engagement and a willingness to connect."}],
  'role': 'model'},
 'finish_reason': 'STOP',
 'safety_ratings': [{'category': 'HARM_CATEGORY_HATE_SPEECH',
   'probability': 'NEGLIGIBLE',
   'probability_score': 1.6488844e-08,
   'severity': 'HARM_SEVERITY_NEGLIGIBLE'},
  {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT',
   'probability': 'NEGLIGIBLE',
   'probability_score': 3.31581e-11,
   'severity': 'HARM_SEVERITY_NEGLIGIBLE',
   'severity_score': 0.03191562},
  {'category': 'HARM_CATEGORY_HARASSMENT',
   'probability': 'NEGLIGIBLE',
   'probability_score': 1.1745181e-08,
   'severity': 'HARM_SEVERITY_NEGLIGIBLE'},
  {'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT',
   'probability': 'NEGLIGIBLE',
   'probability_score': 1.9542265e-11,
   'severity': 'HARM_SEVERITY_NEGLIGIBLE',
   'severity_scor

#### Understanding the safety ratings: category and probability

You can see the safety ratings, including each `category` type and its associated `probability` label.

The `category` types include:

* Hate speech: `HARM_CATEGORY_HATE_SPEECH`
* Dangerous content: `HARM_CATEGORY_DANGEROUS_CONTENT`
* Harassment: `HARM_CATEGORY_HARASSMENT`
* Sexually explicit statements: `HARM_CATEGORY_SEXUALLY_EXPLICIT`

The `probability` labels are:

* `NEGLIGIBLE` - content has a negligible probability of being unsafe
* `LOW` - content has a low probability of being unsafe
* `MEDIUM` - content has a medium probability of being unsafe
* `HIGH` - content has a high probability of being unsafe

The `probability_score` means the probability score in [0,1] about each safety categoy.<br>
Here you should be seeing very low values.

Try a prompt that might trigger one of these categories:

In [6]:
impolite_prompt = "Write a list of 5 disrespectful things that I might say to the universe after stubbing my toe in the dark:"

response = client.models.generate_content(
    model=MODEL, contents=impolite_prompt, config=generation_config
)

response.candidates[0].to_json_dict()

{'content': {'parts': [{'text': 'Here are 5 disrespectful things you might mutter to the universe after stubbing your toe in the dark:\n\n1.  "Oh, *that\'s* original, Universe. Real creative cosmic prank, truly." (Heavy sarcasm, implying the universe is unoriginal and childish)\n2.  "Was that absolutely necessary, Universe? Was my morning *too* pleasant for you?" (Accusatory, implying the universe intentionally seeks to ruin your day)\n3.  "You know what, Universe? You\'re really dropping the ball on the \'good times\' front today." (Condescending, as if the universe has a job it\'s failing at)\n4.  "Great job, Universe. Really showing your master plan with this one." (More sarcasm, questioning the universe\'s intelligence and grand design)\n5.  "Thanks for the reminder, oh great cosmic puppet master, that I\'m just a plaything for your clumsy antics." (Direct insult, diminishing its power by calling its actions "clumsy antics")'}],
  'role': 'model'},
 'finish_reason': 'STOP',
 'avg_l

Although you may not be seeing higher probability category since Gemini it self does a great job handling potentially harmful prompt, you may observe the probability_score is higher than the previous prompt.

### Defining thresholds for safety ratings

You may want to adjust the default safety filter thresholds depending on your business policies or use case. The Gemini provides you a way to pass in a threshold for each category.

The list below shows the possible threshold labels:

* `BLOCK_ONLY_HIGH` - block when high probability of unsafe content is detected
* `BLOCK_MEDIUM_AND_ABOVE` - block when medium or high probablity of content is detected
* `BLOCK_LOW_AND_ABOVE` - block when low, medium, or high probability of unsafe content is detected
* `BLOCK_NONE` - always show, regardless of probability of unsafe content

#### Set safety thresholds
Below, the safety thresholds have been set to the most sensitive threshold: `BLOCK_LOW_AND_ABOVE`

In [7]:
generation_config = GenerateContentConfig(
    safety_settings=[
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
        SafetySetting(
            category=HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
        ),
    ]
)

#### Test thresholds

Here you will reuse the impolite prompt from earlier together with the most sensitive safety threshold. It should block the response even with the `LOW` probability label.

Try multiple times until you see a blocked response.

In [8]:
impolite_prompt = "Write a list of 5 disrespectful things that I might say to the universe after stubbing my toe in the dark:"

response = client.models.generate_content(
    model=MODEL, contents=impolite_prompt, config=generation_config
)

response.candidates[0].to_json_dict()

{'content': {'parts': [{'text': 'Here are 5 disrespectful things you might yell at the universe after stubbing your toe in the dark:\n\n1.  "Oh, *this* is what billions of years of cosmic evolution leads to? A perfectly placed furniture trap? Bravo, you celestial genius!"\n2.  "Thanks for the reminder, Universe, that you clearly have a sick sense of humor and nothing better to do than trip mortals in the dark."\n3.  "Seriously? Is this the best you\'ve got? Because if this is your idea of a grand lesson, you need to go back to cosmic kindergarten."\n4.  "You know what, Universe? I\'m starting to think you just enjoy watching us suffer. Little petty sadist, aren\'t you?"\n5.  "Couldn\'t you have, I don\'t know, put a *light switch* closer? Or maybe *moved the laws of physics* for one second? Incompetent much?"'}],
  'role': 'model'},
 'finish_reason': 'STOP',
 'avg_logprobs': -2.1820248413085936,
 'safety_ratings': [{'category': 'HARM_CATEGORY_HATE_SPEECH',
   'probability': 'NEGLIGIBLE

This notebook is based on [Thu Ya Kyaw](https://github.com/iamthuya)'s work.<br>
https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/responsible-ai/gemini_safety_ratings.ipynb

Copyright 2024 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License